# PS1 — Device / Bus / Serial mapping + device–incident export

Builds one **CSV** file with device details (VALIDATOR / TVM / GATE) plus incident fields — one row per `DEVICE_ID`.

**Run order:** Cell 0 → 1 → 2 → 3 → 4 → 5 → 6

Output: `ps1_validator_device_map/device_incidents.csv`

In [ ]:
# CELL 0 — deps
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "pyarrow", "s3fs",
])
print("OK")

In [ ]:
# CELL 1 — config (edit these if needed)
import os

import pandas as pd

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
BRONZE_BUCKET = "cubic-mars-pm-s3-datalake-dev-bronze-170202974600"
SILVER_PREFIX = "chicago/silver"
BRONZE_PREFIX = "chicago/bronze"
SILVER_BASE = f"s3://{GOLD_BUCKET}/{SILVER_PREFIX}"
BRONZE_BASE = f"s3://{BRONZE_BUCKET}/{BRONZE_PREFIX}"

OUT_DIR = "ps1_validator_device_map"
OUTPUT_BASENAME = "device_incidents"

# Device categories for the bus/serial map (VALIDATOR only — BUS_ID lives here)
DEVICE_MAP_CATEGORIES = ["VALIDATOR"]
# Device categories for device_incidents export
DEVICE_INCIDENT_CATEGORIES = ["VALIDATOR", "TVM", "GATE"]

# Device detail columns (same as device_level / bus-serial map)
DEVICE_DETAIL_COLUMNS = [
    "DEVICE_ID",
    "DEVICE_KEY",
    "DEVICE_NAME",
    "BUS_ID",
    "bus_device_flag",
    "SERIAL_NUMBER",
    "DEVICE_SERIAL_NUMBER",
    "COMPONENT_SERIAL_NBR",
    "COMPONENT_TYPE",
    "FACILITY_ID",
    "FACILITY_NAME",
    "OPERATOR_ID",
    "OPERATOR_NAME",
    "mars_device_category",
    "DEVICE_TYPE_NAME",
    "TRANSIT_MODE_NAME",
]

# Incident summary columns (one row per DEVICE_ID)
INCIDENT_SUMMARY_COLUMNS = [
    "incident_number",
    "incident_sys_id",
    "cmdb_ci_sys_id",
    "opened_at",
    "closed_at",
    "all_incident_numbers",
    "all_incident_sys_ids",
    "all_cmdb_ci_sys_ids",
    "incident_count",
]

INCIDENT_OUTPUT_COLUMNS = DEVICE_DETAIL_COLUMNS + INCIDENT_SUMMARY_COLUMNS

# True = only mobile bus validators (bus_device_flag=True)
BUS_ONLY = False
# True = keep SCD2 historical dim_device rows (usually leave False)
INCLUDE_HISTORICAL = False

S3_OPTS = {"client_kwargs": {"region_name": os.environ.get("AWS_DEFAULT_REGION", "us-east-1")}}

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Silver base : {SILVER_BASE}")
print(f"Out dir     : {os.path.abspath(OUT_DIR)}")
print(f"DEVICE_MAP_CATEGORIES={DEVICE_MAP_CATEGORIES}")
print(f"DEVICE_INCIDENT_CATEGORIES={DEVICE_INCIDENT_CATEGORIES}")
print(f"BUS_ONLY={BUS_ONLY}  INCLUDE_HISTORICAL={INCLUDE_HISTORICAL}")

In [ ]:
# CELL 2 — helpers + load dim_device / hw_config_current

def _read_parquet(uri: str) -> pd.DataFrame:
    print(f"Reading {uri} ...")
    try:
        return pd.read_parquet(uri, storage_options=S3_OPTS)
    except TypeError:
        return pd.read_parquet(uri)


def _pick(df: pd.DataFrame, cols):
    return [c for c in cols if c in df.columns]


def load_devices(
    silver_base: str,
    categories,
    current_only: bool = True,
    label: str = "devices",
) -> pd.DataFrame:
    dim = _read_parquet(f"{silver_base}/dim_device")
    if current_only and "is_current" in dim.columns:
        dim = dim[dim["is_current"] == True].copy()  # noqa: E712
    elif current_only and "CURRENT_FLAG" in dim.columns:
        dim = dim[dim["CURRENT_FLAG"].astype(str).isin(["1", "True", "true"])].copy()

    if "mars_device_category" not in dim.columns:
        raise RuntimeError("dim_device missing mars_device_category")

    cats = {str(c).upper().strip() for c in categories}
    dim = dim[dim["mars_device_category"].astype(str).str.upper().str.strip().isin(cats)].copy()
    if dim.empty:
        raise RuntimeError(f"No {sorted(cats)} rows found in dim_device")

    keep = _pick(dim, [
        "DEVICE_ID", "DEVICE_KEY", "DEVICE_NAME", "DEVICE_SERIAL_NUMBER",
        "BUS_ID", "bus_device_flag", "FACILITY_ID", "FACILITY_NAME",
        "OPERATOR_ID", "OPERATOR_NAME", "mars_device_category",
        "DEVICE_TYPE_NAME", "TRANSIT_MODE_NAME",
    ])
    out = dim[keep].drop_duplicates(subset=["DEVICE_ID"] if "DEVICE_ID" in keep else ["DEVICE_KEY"])
    print(f"{label} ({', '.join(sorted(cats))}): {len(out):,}")
    return out


def load_validator_devices(silver_base: str, current_only: bool = True) -> pd.DataFrame:
    return load_devices(
        silver_base,
        categories=DEVICE_MAP_CATEGORIES,
        current_only=current_only,
        label="VALIDATOR devices (dim_device)",
    )


def load_hw_serials(silver_base: str) -> pd.DataFrame:
    try:
        hw = _read_parquet(f"{silver_base}/hw_config_current")
    except Exception as exc:
        print(f"[WARN] hw_config_current not readable: {exc}")
        return pd.DataFrame()

    serial_col = next(
        (c for c in ("COMPONENT_SERIAL_NBR", "SERIAL_NUMBER", "serial_number", "COMPONENT_SERIAL") if c in hw.columns),
        None,
    )
    if serial_col is None:
        print("[WARN] hw_config_current has no serial column — device-level serial only")
        return pd.DataFrame()

    type_col = next(
        (c for c in ("COMPONENT_DESCRIPTION", "COMPONENT_TYPE", "COMPONENT_PART_NBR") if c in hw.columns),
        None,
    )
    key_cols = _pick(hw, ["DEVICE_KEY", "DEVICE_ID"])
    if not key_cols:
        print("[WARN] hw_config_current missing DEVICE_KEY/DEVICE_ID")
        return pd.DataFrame()

    cols = key_cols + [serial_col] + ([type_col] if type_col else [])
    hw = hw[cols].dropna(subset=[serial_col]).copy()
    hw = hw.rename(columns={serial_col: "COMPONENT_SERIAL_NBR"})
    if type_col and type_col != "COMPONENT_TYPE":
        hw = hw.rename(columns={type_col: "COMPONENT_TYPE"})
    elif type_col is None:
        hw["COMPONENT_TYPE"] = None

    dedupe_keys = [c for c in ["DEVICE_KEY", "DEVICE_ID", "COMPONENT_SERIAL_NBR"] if c in hw.columns]
    hw = hw.drop_duplicates(subset=dedupe_keys)
    print(f"hw_config serial rows: {len(hw):,}")
    return hw


devices = load_validator_devices(SILVER_BASE, current_only=not INCLUDE_HISTORICAL)
hw = load_hw_serials(SILVER_BASE)
display(devices.head(3))
print("device columns:", list(devices.columns))

In [ ]:
# CELL 3 — build DEVICE_ID / BUS_ID / serial mapping

def build_mapping(devices: pd.DataFrame, hw: pd.DataFrame, bus_only: bool) -> pd.DataFrame:
    if bus_only and "bus_device_flag" in devices.columns:
        flag = devices["bus_device_flag"]
        devices = devices[(flag == True) | (flag.astype(str).str.lower().isin(["1", "true"]))].copy()  # noqa: E712
        print(f"After bus_device_flag=True filter: {len(devices):,}")

    if hw.empty:
        mapped = devices.copy()
        mapped["COMPONENT_SERIAL_NBR"] = pd.NA
        mapped["COMPONENT_TYPE"] = pd.NA
    else:
        if "DEVICE_KEY" in devices.columns and "DEVICE_KEY" in hw.columns:
            mapped = devices.merge(hw, on="DEVICE_KEY", how="left")
            if "DEVICE_ID_y" in mapped.columns:
                mapped = mapped.drop(columns=["DEVICE_ID_y"]).rename(columns={"DEVICE_ID_x": "DEVICE_ID"})
        elif "DEVICE_ID" in devices.columns and "DEVICE_ID" in hw.columns:
            mapped = devices.merge(hw, on="DEVICE_ID", how="left")
            if "DEVICE_KEY_y" in mapped.columns:
                mapped = mapped.drop(columns=["DEVICE_KEY_y"]).rename(columns={"DEVICE_KEY_x": "DEVICE_KEY"})
        else:
            mapped = devices.copy()
            mapped["COMPONENT_SERIAL_NBR"] = pd.NA
            mapped["COMPONENT_TYPE"] = pd.NA

    ds = mapped["DEVICE_SERIAL_NUMBER"] if "DEVICE_SERIAL_NUMBER" in mapped.columns else pd.Series(pd.NA, index=mapped.index)
    cs = mapped["COMPONENT_SERIAL_NBR"] if "COMPONENT_SERIAL_NBR" in mapped.columns else pd.Series(pd.NA, index=mapped.index)
    mapped["SERIAL_NUMBER"] = cs.where(cs.notna() & (cs.astype(str).str.strip() != ""), ds)

    preferred = [
        "DEVICE_ID", "DEVICE_KEY", "DEVICE_NAME", "BUS_ID", "bus_device_flag",
        "SERIAL_NUMBER", "DEVICE_SERIAL_NUMBER", "COMPONENT_SERIAL_NBR", "COMPONENT_TYPE",
        "FACILITY_ID", "FACILITY_NAME", "OPERATOR_ID", "OPERATOR_NAME",
        "mars_device_category", "DEVICE_TYPE_NAME", "TRANSIT_MODE_NAME",
    ]
    cols = [c for c in preferred if c in mapped.columns] + [c for c in mapped.columns if c not in preferred]
    mapped = mapped[cols].sort_values(
        by=[c for c in ["BUS_ID", "DEVICE_ID", "COMPONENT_SERIAL_NBR"] if c in mapped.columns],
        kind="mergesort",
    ).reset_index(drop=True)
    return mapped


mapped = build_mapping(devices, hw, bus_only=BUS_ONLY)

n_bus = int(mapped["BUS_ID"].notna().sum()) if "BUS_ID" in mapped.columns else 0
n_ser = int(mapped["SERIAL_NUMBER"].notna().sum()) if "SERIAL_NUMBER" in mapped.columns else 0
n_dev = mapped["DEVICE_ID"].nunique() if "DEVICE_ID" in mapped.columns else len(mapped)
print(
    f"Mapping summary: devices={n_dev:,}  rows={len(mapped):,}  "
    f"BUS_ID populated={n_bus:,}  SERIAL_NUMBER populated={n_ser:,}"
)
display(mapped.head(20))

In [ ]:
# CELL 4 — quick checks
print("Null rates (key cols):")
for c in ["DEVICE_ID", "BUS_ID", "SERIAL_NUMBER", "COMPONENT_SERIAL_NBR", "DEVICE_SERIAL_NUMBER", "bus_device_flag"]:
    if c in mapped.columns:
        print(f"  {c:<22} null={mapped[c].isna().mean():.1%}")

if "BUS_ID" in mapped.columns:
    print("\nTop BUS_ID by device count:")
    display(
        mapped.groupby("BUS_ID", dropna=False)["DEVICE_ID"].nunique()
        .sort_values(ascending=False).head(15).rename("n_devices").to_frame()
    )

In [ ]:
# CELL 5 — build device–incident mapping (VALIDATOR + TVM + GATE)
# Grain: one row per DEVICE_ID (latest incident + comma-separated lists)

def load_incident_task_ci_link(silver_base: str) -> pd.DataFrame:
    try:
        inc = _read_parquet(f"{silver_base}/incident_task_ci_link")
    except Exception as exc:
        print(f"[WARN] incident_task_ci_link not readable: {exc}")
        return pd.DataFrame()
    if "DEVICE_ID" in inc.columns:
        inc = inc[inc["DEVICE_ID"].notna()].copy()
    print(f"incident_task_ci_link rows (DEVICE_ID populated): {len(inc):,}")
    return inc


def load_incident_history(silver_base: str) -> pd.DataFrame:
    try:
        ih = _read_parquet(f"{silver_base}/incident_history")
    except Exception as exc:
        print(f"[WARN] incident_history not readable: {exc}")
        return pd.DataFrame()
    print(f"incident_history rows: {len(ih):,}")
    return ih


def load_task_ci_cmdb_map(bronze_base: str) -> pd.DataFrame:
    """task_ci.sys_id -> ci_item_sys_id (CMDB CI for the linked bus asset)."""
    uri = f"{bronze_base}/servicenow_task_ci"
    try:
        tc = _read_parquet(uri)
    except Exception as exc:
        print(f"[WARN] servicenow_task_ci not readable ({exc}) — cmdb_ci_sys_id may be blank for VALIDATOR")
        return pd.DataFrame()

    id_col = next((c for c in ("sys_id", "task_ci_sys_id") if c in tc.columns), None)
    ci_col = next((c for c in ("ci_item_sys_id", "cmdb_ci_sys_id") if c in tc.columns), None)
    if id_col is None or ci_col is None:
        print("[WARN] servicenow_task_ci missing sys_id/ci_item_sys_id")
        return pd.DataFrame()

    out = tc[[id_col, ci_col]].dropna(subset=[id_col]).drop_duplicates(subset=[id_col])
    out = out.rename(columns={id_col: "task_ci_sys_id", ci_col: "cmdb_ci_sys_id"})
    print(f"task_ci CMDB map rows: {len(out):,}")
    return out


def _norm_id(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    s = s.where(s.notna(), pd.NA)
    s = s.str.upper()
    return s.mask(s.isin(["NAN", "NONE", "NAT", ""]), pd.NA)


def _norm_fleet(ids) -> set:
    return set(_norm_id(pd.Series(list(ids))).dropna().unique())


def _prepare_incident_keys(inc: pd.DataFrame) -> pd.DataFrame:
    """Normalize join/dedupe keys so one row per DEVICE_ID × incident."""
    out = inc.copy()
    if "DEVICE_ID" in out.columns:
        out["DEVICE_ID"] = _norm_id(out["DEVICE_ID"])

    # Coalesce incident id fields (task_ci_link uses linked_* aliases)
    if "linked_incident_number" in out.columns:
        base_num = out["incident_number"] if "incident_number" in out.columns else pd.Series(pd.NA, index=out.index)
        out["incident_number"] = base_num.fillna(out["linked_incident_number"])
    if "incident_number" in out.columns:
        out["incident_number"] = out["incident_number"].astype(str).str.strip().replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})

    if "linked_incident_sys_id" in out.columns:
        base_sys = out["incident_sys_id"] if "incident_sys_id" in out.columns else pd.Series(pd.NA, index=out.index)
        out["incident_sys_id"] = base_sys.fillna(out["linked_incident_sys_id"])

    # Canonical dedupe key: incident_number preferred, else incident_sys_id
    num = out["incident_number"] if "incident_number" in out.columns else pd.Series(pd.NA, index=out.index)
    sys = out["incident_sys_id"] if "incident_sys_id" in out.columns else pd.Series(pd.NA, index=out.index)
    out["_incident_key"] = num.where(num.notna() & (num.astype(str).str.strip() != ""), sys.astype(str).str.strip())
    out["_incident_key"] = out["_incident_key"].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})

    before = len(out)
    out = out[out["DEVICE_ID"].notna() & out["_incident_key"].notna()].copy()
    if len(out) < before:
        print(f"Dropped {before - len(out):,} rows missing DEVICE_ID or incident id")
    return out


def _comma_unique(values, ordered=False) -> str:
    seen = set()
    out = []
    seq = values if ordered else sorted(values, key=lambda x: str(x))
    for v in seq:
        if v is None or (isinstance(v, float) and pd.isna(v)):
            continue
        s = str(v).strip()
        if not s or s.lower() in ("nan", "none", "nat") or s in seen:
            continue
        seen.add(s)
        out.append(s)
    return ", ".join(out)


def _dedupe_incident_events(inc: pd.DataFrame) -> pd.DataFrame:
    """One row per DEVICE_ID × incident (intermediate)."""
    if inc.empty:
        return inc

    inc = _prepare_incident_keys(inc)
    sort_cols = [c for c in ("opened_at", "closed_at", "incident_updated_on", "task_ci_sys_id") if c in inc.columns]
    if sort_cols:
        inc = inc.sort_values(sort_cols, na_position="last", kind="mergesort")

    before = len(inc)
    inc = inc.drop_duplicates(subset=["DEVICE_ID", "_incident_key"], keep="first")
    if len(inc) < before:
        print(f"Deduped {before - len(inc):,} duplicate DEVICE_ID × incident rows")
    return inc


def build_device_level_lookup(devices: pd.DataFrame, hw: pd.DataFrame) -> pd.DataFrame:
    """One row per DEVICE_ID with full device / bus / serial details."""
    mapped_all = build_mapping(devices, hw, bus_only=False)
    if "DEVICE_ID" not in mapped_all.columns:
        return pd.DataFrame(columns=DEVICE_DETAIL_COLUMNS)

    out = (
        mapped_all.sort_values(
            by=[c for c in ["COMPONENT_SERIAL_NBR"] if c in mapped_all.columns],
            na_position="last",
            kind="mergesort",
        )
        .drop_duplicates(subset=["DEVICE_ID"], keep="first")
        .copy()
    )
    if "SERIAL_NUMBER" in mapped_all.columns:
        best_serial = (
            mapped_all.groupby("DEVICE_ID", as_index=False)["SERIAL_NUMBER"]
            .agg(lambda s: next((x for x in s if pd.notna(x) and str(x).strip()), pd.NA))
        )
        out = out.drop(columns=["SERIAL_NUMBER"], errors="ignore").merge(best_serial, on="DEVICE_ID", how="left")

    out["DEVICE_ID"] = _norm_id(out["DEVICE_ID"])
    for c in DEVICE_DETAIL_COLUMNS:
        if c not in out.columns:
            out[c] = pd.NA
    return out[DEVICE_DETAIL_COLUMNS].drop_duplicates(subset=["DEVICE_ID"])


def _collapse_to_device_level(inc: pd.DataFrame) -> pd.DataFrame:
    """One row per DEVICE_ID — latest incident + comma-separated lists (incident cols only)."""
    if inc.empty:
        return pd.DataFrame(columns=["DEVICE_ID"] + INCIDENT_SUMMARY_COLUMNS)

    inc = _dedupe_incident_events(inc)
    sort_cols = [c for c in ("opened_at", "closed_at") if c in inc.columns]
    if sort_cols:
        inc = inc.sort_values(sort_cols, ascending=False, na_position="last", kind="mergesort")

    list_cols = {
        "all_incident_numbers": "incident_number",
        "all_incident_sys_ids": "incident_sys_id",
        "all_cmdb_ci_sys_ids": "cmdb_ci_sys_id",
    }

    rows = []
    for device_id, grp in inc.groupby("DEVICE_ID", sort=False):
        grp_sorted = grp.sort_values(sort_cols, ascending=False, na_position="last", kind="mergesort") if sort_cols else grp
        latest = grp_sorted.iloc[0]
        row = {
            "DEVICE_ID": device_id,
            "incident_number": latest.get("incident_number"),
            "incident_sys_id": latest.get("incident_sys_id"),
            "cmdb_ci_sys_id": latest.get("cmdb_ci_sys_id"),
            "opened_at": latest.get("opened_at"),
            "closed_at": latest.get("closed_at"),
            "incident_count": int(grp["_incident_key"].nunique()),
        }
        grp_list = grp.sort_values(sort_cols, ascending=True, na_position="last", kind="mergesort") if sort_cols else grp
        for out_col, src_col in list_cols.items():
            row[out_col] = _comma_unique(grp_list[src_col].tolist(), ordered=True) if src_col in grp_list.columns else ""
        rows.append(row)

    return pd.DataFrame(rows)


def _attach_device_details(device_details: pd.DataFrame, incident_summary: pd.DataFrame) -> pd.DataFrame:
    """Left-join incident summary onto full device lookup — one row per fleet DEVICE_ID."""
    if device_details.empty:
        return pd.DataFrame(columns=INCIDENT_OUTPUT_COLUMNS)

    out = device_details.copy()
    if not incident_summary.empty:
        out = out.merge(incident_summary, on="DEVICE_ID", how="left")

    out["incident_count"] = out["incident_count"].fillna(0).astype(int)
    for c in ("all_incident_numbers", "all_incident_sys_ids", "all_cmdb_ci_sys_ids"):
        if c in out.columns:
            out[c] = out[c].fillna("")
        else:
            out[c] = ""

    for c in INCIDENT_OUTPUT_COLUMNS:
        if c not in out.columns:
            out[c] = pd.NA
    return out[INCIDENT_OUTPUT_COLUMNS].sort_values(
        by=[c for c in ("mars_device_category", "DEVICE_ID") if c in out.columns],
        kind="mergesort",
    ).reset_index(drop=True)


def build_validator_incidents(
    incidents: pd.DataFrame,
    device_lookup: pd.DataFrame,
    cmdb_map: pd.DataFrame,
    fleet_device_ids=None,
) -> pd.DataFrame:
    if incidents.empty:
        return pd.DataFrame()

    inc = incidents.copy()
    inc["DEVICE_ID"] = _norm_id(inc["DEVICE_ID"])
    if fleet_device_ids is not None:
        inc = inc[inc["DEVICE_ID"].isin(_norm_fleet(fleet_device_ids))].copy()
        print(f"VALIDATOR incidents after fleet filter: {len(inc):,} rows")

    lookup_cols = [
        c for c in (
            "DEVICE_ID", "BUS_ID", "DEVICE_KEY", "FACILITY_ID", "FACILITY_NAME",
            "OPERATOR_ID", "mars_device_category",
        ) if c in device_lookup.columns
    ]
    lookup = device_lookup[lookup_cols].copy()
    lookup["DEVICE_ID"] = _norm_id(lookup["DEVICE_ID"])
    lookup = lookup.drop_duplicates(subset=["DEVICE_ID"])
    drop_cols = [c for c in ("BUS_ID", "DEVICE_KEY", "FACILITY_ID", "FACILITY_NAME", "OPERATOR_ID", "mars_device_category") if c in inc.columns]
    inc = inc.drop(columns=drop_cols, errors="ignore").merge(lookup, on="DEVICE_ID", how="left")

    if not cmdb_map.empty and "task_ci_sys_id" in inc.columns:
        inc = inc.merge(cmdb_map, on="task_ci_sys_id", how="left")
    elif "cmdb_ci_sys_id" not in inc.columns:
        inc["cmdb_ci_sys_id"] = pd.NA

    return _dedupe_incident_events(inc)


def build_tvm_gate_incidents(
    history: pd.DataFrame,
    device_lookup: pd.DataFrame,
    fleet_device_ids=None,
) -> pd.DataFrame:
    if history.empty:
        return pd.DataFrame()

    ih = history.copy()
    if "wm_asset" in ih.columns:
        ih["DEVICE_ID"] = _norm_id(ih["wm_asset"])
    elif "DEVICE_ID" in ih.columns:
        ih["DEVICE_ID"] = _norm_id(ih["DEVICE_ID"])
    else:
        print("[WARN] incident_history has no wm_asset/DEVICE_ID — skipping TVM/GATE")
        return pd.DataFrame()

    rename = {
        "opened_dtm": "opened_at",
        "resolved_dtm": "resolved_at",
        "closed_dtm": "closed_at",
    }
    ih = ih.rename(columns={k: v for k, v in rename.items() if k in ih.columns})
    if "incident_sys_id" not in ih.columns:
        ih["incident_sys_id"] = pd.NA

    if fleet_device_ids is not None:
        ih = ih[ih["DEVICE_ID"].isin(_norm_fleet(fleet_device_ids))].copy()
        print(f"TVM/GATE incidents after fleet filter: {len(ih):,} rows")

    lookup_cols = [
        c for c in (
            "DEVICE_ID", "BUS_ID", "DEVICE_KEY", "FACILITY_ID", "FACILITY_NAME",
            "OPERATOR_ID", "mars_device_category",
        ) if c in device_lookup.columns
    ]
    lookup = device_lookup[lookup_cols].copy()
    lookup["DEVICE_ID"] = _norm_id(lookup["DEVICE_ID"])
    lookup = lookup.drop_duplicates(subset=["DEVICE_ID"])
    drop_cols = [c for c in ("BUS_ID", "DEVICE_KEY", "FACILITY_ID", "FACILITY_NAME", "OPERATOR_ID", "mars_device_category") if c in ih.columns]
    ih = ih.drop(columns=drop_cols, errors="ignore").merge(lookup, on="DEVICE_ID", how="left")

    return _dedupe_incident_events(ih)


incident_fleet = load_devices(
    SILVER_BASE,
    categories=DEVICE_INCIDENT_CATEGORIES,
    current_only=not INCLUDE_HISTORICAL,
    label="Incident fleet devices",
)
hw_all = load_hw_serials(SILVER_BASE)
incident_device_details = build_device_level_lookup(incident_fleet, hw_all)
print(f"Device detail lookup: {len(incident_device_details):,} rows")

device_lookup = incident_fleet.drop_duplicates(subset=["DEVICE_ID"])
fleet_ids = set(device_lookup["DEVICE_ID"].dropna().unique())

validator_ids = set(
    device_lookup.loc[
        device_lookup["mars_device_category"].astype(str).str.upper() == "VALIDATOR", "DEVICE_ID"
    ].dropna()
) if "mars_device_category" in device_lookup.columns else fleet_ids

tvm_gate_ids = fleet_ids - validator_ids

incidents_validator_raw = load_incident_task_ci_link(SILVER_BASE)
incidents_history_raw = load_incident_history(SILVER_BASE)
cmdb_map = load_task_ci_cmdb_map(BRONZE_BASE)

validator_incidents = build_validator_incidents(
    incidents_validator_raw, device_lookup, cmdb_map, fleet_device_ids=validator_ids,
)
tvm_gate_incidents = build_tvm_gate_incidents(
    incidents_history_raw, device_lookup, fleet_device_ids=tvm_gate_ids,
)

parts = [df for df in (validator_incidents, tvm_gate_incidents) if not df.empty]
incident_events = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
incident_summary = _collapse_to_device_level(incident_events)
device_incidents = _attach_device_details(incident_device_details, incident_summary)

if device_incidents.empty:
    print("[WARN] device_incidents is empty — check S3 paths / silver exports")
else:
    n_dev = len(device_incidents)
    n_with_inc = int((device_incidents["incident_count"].fillna(0) > 0).sum())
    total_inc = int(device_incidents["incident_count"].fillna(0).sum())
    print(
        f"device_incidents summary: rows={n_dev:,} (one per DEVICE_ID)  "
        f"devices_with_incidents={n_with_inc:,}  total_incidents={total_inc:,}"
    )
    if "mars_device_category" in device_incidents.columns:
        print("By device category:")
        display(
            device_incidents.groupby("mars_device_category", dropna=False)
            .agg(n_devices=("DEVICE_ID", "size"), total_incidents=("incident_count", "sum"))
            .sort_values("total_incidents", ascending=False)
        )
    dup_n = int(device_incidents.duplicated(subset=["DEVICE_ID"], keep=False).sum())
    if dup_n:
        print(f"[WARN] {dup_n} rows still duplicate on DEVICE_ID")
    else:
        print("Dedupe check: PASS — unique DEVICE_ID per row")
    display(device_incidents.head(20))

In [ ]:
# CELL 6 — write single CSV
csv_path = os.path.join(OUT_DIR, f"{OUTPUT_BASENAME}.csv")

if device_incidents.empty:
    print("[WARN] Nothing to write — device_incidents is empty")
else:
    device_incidents.to_csv(csv_path, index=False)
    print(f"Wrote CSV: {os.path.abspath(csv_path)}  ({len(device_incidents):,} rows)")

print("Done.")

In [ ]:
# CELL 7 (optional) — download CSV in Jupyter
from IPython.display import FileLink, display

if os.path.exists(csv_path):
    display(FileLink(csv_path))